# 巨大数ポーカー AI — Colab で学習する

**やることは「ランタイム → すべてのセルを実行」だけ。**（`Ctrl+F9`）

設定は下の ① のフォームで選べます。全部そのままでも動きます。

---

### 手元のGUIとのつなぎ方

```
手元GUI「Colabで学習を開始」ボタン → このノートが開く → Ctrl+F9
                                          ↓
                              学習（Drive に自動保存）
                                          ↓
                        最後のセルが結果を1つの zip でダウンロード
                                          ↓
                  手元GUI の「結果を取り込む」に zip をドラッグ
```

Google Drive API の認証設定（credentials.json など）は要りません。

> **ランタイムについて**: この学習は Node 側（ゲームのルールと巨大数エンジン）が
> 律速なので、**GPU にしてもほとんど速くなりません**。CPU のままで構いません。
> むしろ CPU ランタイムのほうが割り当てが早いことが多いです。


In [ ]:
#@title ① 設定（ここだけ触ればOK）{ display-mode: 'form' }

#@markdown 相手にする「人間」の計算力。手元のGUIのモード選択と同じもの。
レベル = 'skilled' #@param ['novice', 'casual', 'skilled', 'expert', 'master']
#@markdown 何世代まわすか。1世代あたり数秒〜十数秒。
世代数 = 100 #@param {type:'slider', min:10, max:500, step:10}
#@markdown 並列する卓の数。多いほど1世代が安定するがメモリを食う。
並列環境数 = 256 #@param {type:'slider', min:32, max:1024, step:32}
#@markdown 何世代ごとに強さを測るか。
評価の間隔 = 5 #@param {type:'slider', min:1, max:20, step:1}
#@markdown リポジトリ（フォークして使うときは書き換える）
リポジトリ = 'https://github.com/tomikan1208-code/Huge_Number_Poker.git' #@param {type:'string'}
ブランチ = 'main' #@param {type:'string'}

import os
CFG = dict(level=レベル, gens=世代数, envs=並列環境数,
           eval_every=評価の間隔, repo=リポジトリ, branch=ブランチ)
for k, v in CFG.items():
    print(f'{k:12} {v}')


In [ ]:
#@title ② 準備（Drive・コード取得・環境の動作確認）

# Drive は「途中で切れても続きから再開する」ために使う。
# train.py が世代ごとにチェックポイントとログをここへ写す。
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/huge_number_poker_checkpoints'
os.makedirs(DATA_DIR, exist_ok=True)
print('永続フォルダ:', DATA_DIR)

# ! 行に埋める値は、クォートを含まない素の変数にしておく。
# {CFG['branch']} のように書くと IPython の展開でクォートが衝突して壊れる。
REPO   = CFG['repo']
BRANCH = CFG['branch']
LEVEL  = CFG['level']

# GitHub から直接取ってくる。Drive にフォルダをアップロードする必要はない。
%cd /content
if os.path.isdir('Huge_Number_Poker'):
    !cd Huge_Number_Poker && git fetch -q origin $BRANCH && git checkout -q $BRANCH && git reset --hard -q origin/$BRANCH
else:
    !git clone -q -b $BRANCH $REPO Huge_Number_Poker
%cd /content/Huge_Number_Poker
!git log -1 --format='取得したコミット: %h %s'

!pip install -q torch numpy
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs)

# 環境が単体で動くか先に確かめる。ここで落ちるなら学習しても無駄。
print('\n--- 環境の動作確認 ---')
!node train/env_server.js --selfplay 200 --level $LEVEL | head -12


In [ ]:
#@title ③ 学習（■ 停止ボタンで安全に中断できる）

# 環境変数で渡す。手元のGUIから起動するときとまったく同じ経路。
import os
os.environ['HNP_LEVEL']      = CFG['level']
os.environ['HNP_MAX_GENS']   = str(CFG['gens'])
os.environ['HNP_NUM_ENVS']   = str(CFG['envs'])
os.environ['HNP_EVAL_EVERY'] = str(CFG['eval_every'])

# 中断してもチェックポイントは世代ごとに Drive へ写っているので、
# 次回 ② を実行すれば続きから再開する。
!python train/train.py


In [ ]:
#@title ④ 結果をまとめてダウンロード（手元GUIに放り込む用）

# ログ・重み・**チェックポイント** を1つの zip にする。
# .pt を入れるのが要点。これが無いと手元に持ち帰っても「グラフが繋がるだけ」で、
# 学習は最初からになる。ネットワークは小さいので .pt も数百KB程度。
import glob, os, shutil, zipfile, datetime

stamp = datetime.datetime.now().strftime('%m%d_%H%M')
out = '/content/hnp_colab_' + CFG['level'] + '_' + stamp + '.zip'
targets = sorted(set(
    glob.glob('train/models/*_log.json')
    + glob.glob('train/models/*.pt')
    + glob.glob('models/policy_*.json')))

with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for pth in targets:
        z.write(pth, os.path.basename(pth))
        print('  + %-40s %6.1f KB' % (pth, os.path.getsize(pth) / 1024))

# Drive にも同じものを置く（ランタイムが切れても残る）
for pth in targets:
    shutil.copy2(pth, os.path.join(DATA_DIR, os.path.basename(pth)))
print('\nDrive にも保存:', DATA_DIR)

if not targets:
    print('\n⚠ 出力が見つかりません。③ が最後まで走ったか確認してください。')
else:
    print('\n合計 %.1f KB → %s をダウンロードします' % (
        os.path.getsize(out) / 1024, os.path.basename(out)))
    from google.colab import files
    files.download(out)


## 途中で切れたら

**② をもう一度実行するだけ**で続きから再開します。
チェックポイントは世代ごとに Drive（`huge_number_poker_checkpoints`）へ写っているためです。

## 手元のGUIで見る

④ でダウンロードした zip を、手元のコントロールパネルの
**「Colab」タブ → 結果を取り込む** に放り込んでください。
グラフに **点線** で重なって、手元で回した結果と比べられます。

重み（`policy_<レベル>.json`）は取り込んだだけでは反映されません。
同じ画面の **「ゲームに反映」** を押すと、ゲーム本体が読む場所へコピーされます
（上書きになるので、押したときだけ動きます）。

> **注意:** 特徴量（`js/ai-policy.js` の `OBS_DIM`）を変えたら重みは互換性を失います。
> 読み込み時に次元を照合して、合わなければ警告を出して無視します。
> 取り込み画面にも `obs_dim` を表示しているので、そこで確認できます。


In [ ]:
#@title （任意）Colab の中でダッシュボードを開く

# Colab のプロキシ越しなので SSE（リアルタイム更新）が詰まることがある。
# 学習の進行は ③ の [Progress] 行でも追える。
from google.colab import output
import threading, sys
sys.path.insert(0, '/content/Huge_Number_Poker/train')
from dashboard_server import app, PORT

threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=PORT, debug=False, threaded=True),
    daemon=True).start()
output.serve_kernel_port_as_window(PORT)
